# Member Retention Toolkit for Recreational Sports and Membership Organizations
**DSC 680 – Project 2, Milestone 2 | Open, reusable notebook**

This notebook finds members who are **drifting away before they stop renewing**, so staff can reach out while it still matters.
It is built to be reused: you change one settings cell, run all cells, and get (1) a cleaned dataset with a log of every change,
(2) a comparison of four models with honest validation, and (3) a ranked **at-risk list** with plain-language reasons.

> **The data shipped with this notebook is SYNTHETIC.** It is generated by the code below so the tool can be shared publicly with
> no real member's information. It contains deliberate mess (duplicates, inconsistent labels, missing values, post-expiry visits)
> and deliberately planted patterns, so you can check that the tool finds what we know is there. Results on synthetic data
> show that the method works; they say nothing about *your* members until you run it on your own data and validate locally.

### How to use it with your own data (about 15 minutes)
1. **Export two tables** from your membership system: a *members* table (one row per member/account) and a *visits* table (one row per check-in).
2. In the settings cell below, set `USE_DEMO_DATA = False`, point to your two CSV files, and map your column names in `COLS`.
3. Tell the notebook what your status words mean (`CHURNED_WORDS`, `RENEWED_WORDS`) and how your plan names group together (`PLAN_WORDS`).
4. **Remove or leave out names, emails, phones, and addresses.** The notebook also drops columns that look like personal data, but you remain responsible for what you load.
5. Run all cells. Read the **readiness report** (section 3) first: it warns you if your data is too small or too unbalanced to trust.
6. Review `at_risk_list.csv` as a *prompt for friendly outreach by a person*, never as a reason to penalize a member.

| Required column (members) | Meaning |
|---|---|
| `member_id` | Any anonymous ID that matches the visits table |
| `plan` | Membership type (text) |
| `join_date`, `term_end` | Start of membership and end of the current term |
| `status` | Outcome text, such as Renewed / Cancelled / Lapsed. Unclear values are set aside, not guessed |

Visits table: `member_id`, `visit_date`, and (recommended) a unique `visit_id` so true duplicates can be removed without deleting real same-day visits.
Optional numeric member columns known *before* the term ends (for example guest passes) can be listed in `OPTIONAL_NUMERIC`.

In [1]:
import json, os
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (average_precision_score, precision_score, recall_score, f1_score,
                             confusion_matrix, precision_recall_curve)
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

## 1. Settings (the only cell most people need to edit)

In [2]:
CONFIG = dict(
    USE_DEMO_DATA = True,                      # True = build the synthetic demo; False = load your own CSVs below
    MEMBERS_CSV   = "my_members.csv",
    VISITS_CSV    = "my_visits.csv",
    # Map the notebook's names (left) to YOUR column names (right). The demo's own names are shown here as an example.
    COLS = dict(member_id="member_id", plan="membership_type", join_date="join_date", term_end="term_end_date",
                status="status", visit_id="visit_id", visit_date="visit_date"),
    OPTIONAL_NUMERIC = ["guest_passes_used"],  # extra numeric member columns known before term end ([] if none)
    LEAD_DAYS = 30,                            # predict this many days before the term ends (how early staff need warning)
    CHURNED_WORDS = ["cancel", "lapse", "expire", "not renew", "non-renew"],   # status text containing any of these = left
    RENEWED_WORDS = ["renew"],                 # status text containing any of these (and no churn word) = stayed
    PLAN_WORDS = {"month": "Monthly", "annual": "Annual", "year": "Annual", "family": "Family",
                  "student": "Student/Senior", "senior": "Student/Senior"},   # text found in a plan name -> tidy plan name
    PII_PATTERNS = ["email", "phone", "name", "address", "birth", "dob", "ssn", "zip"],  # columns like these are dropped automatically
    FAIRNESS_GROUP = "plan",                   # report model errors for each group in this column
    SEED = 680,
)
SEED = CONFIG["SEED"]
OUT = ""          # write outputs next to the notebook
LOG = {}          # every row-count change is recorded here

## 2. Load the data
The demo builder below creates the two raw tables. Skip it by setting `USE_DEMO_DATA = False`.

**What the synthetic data deliberately contains** (so you can test that the tool finds it):
- *Planted drivers of churn:* low visit frequency, a recent drop in visits, month-to-month plans, short tenure, fewer guest passes.
- *Planted mess:* ~3% duplicate accounts, duplicate check-ins, plan names typed several ways, missing plans and term-end dates,
  missing visit dates, unresolved statuses (Pending/Hold), fake emails and phones, and check-ins that occur **after** the observation date (a leakage trap).

In [3]:
def make_demo_data(N=4000):
    rng = np.random.default_rng(SEED)
    TYPES = np.array(["Monthly", "Annual", "Family", "Student/Senior"])
    mtype = rng.choice(TYPES, size=N, p=[0.38, 0.30, 0.20, 0.12])
    term_end = pd.Timestamp("2025-03-01") + pd.to_timedelta(rng.integers(0, 480, N), unit="D")
    obs_date = term_end - pd.Timedelta(days=30)
    tenure_days = np.clip(rng.gamma(2.0, 200, N), 30, 1800).astype(int)
    join_date = term_end - pd.to_timedelta(tenure_days, unit="D")
    base = {"Monthly": 7.0, "Annual": 9.0, "Family": 11.0, "Student/Senior": 6.0}
    m = np.clip([rng.gamma(2.2, base[t] / 2.2) for t in mtype], 0.3, 30)     # visits per 30 days (latent habit)
    tau = np.clip(rng.lognormal(0, 0.45, N), 0.05, 3)                         # recent change in habit
    guests = rng.poisson(0.6, N)
    logit = (-0.15 - 0.11 * m - 1.0 * (tau - 1) - 0.018 * tenure_days / 30.4 - 0.25 * guests
             + np.select([mtype == "Monthly", mtype == "Annual", mtype == "Family"], [0.55, -0.45, -0.60], 0.20))
    churn_true = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    rows = []
    for i in range(N):
        E = int(np.clip(tenure_days[i] - 30, 0, 335))
        n_early = rng.poisson(m[i] * E / 30); n_rec = rng.poisson(m[i] * tau[i])
        n_post = rng.poisson(m[i] * (0.30 if churn_true[i] else 1.0))
        d = np.concatenate([obs_date[i] - pd.to_timedelta(rng.integers(30, 30 + max(E, 1), n_early), unit="D"),
                            obs_date[i] - pd.to_timedelta(rng.integers(0, 30, n_rec), unit="D"),
                            obs_date[i] + pd.to_timedelta(rng.integers(1, 31, n_post), unit="D")])
        rows.append(pd.DataFrame({"member_id": f"M{100000+i}", "visit_date": d}))
    visits = pd.concat(rows, ignore_index=True)
    visits.insert(0, "visit_id", np.arange(len(visits)))
    visits.loc[rng.random(len(visits)) < 0.005, "visit_date"] = pd.NaT
    visits = pd.concat([visits, visits.sample(frac=0.005, random_state=SEED)])
    tv = {"Monthly": ["Monthly", "monthly", "Month-to-Month ", "MONTHLY"], "Annual": ["Annual", "annual", "Yearly"],
          "Family": ["Family", "family ", "FAMILY"], "Student/Senior": ["Student/Senior", "Senior", "Student"]}
    sv = {0: ["Renewed", "renewed", "RENEWED", "Active - Renewed"], 1: ["Cancelled", "cancelled", "Lapsed", "Expired - Not Renewed"]}
    members = pd.DataFrame({
        "member_id": [f"M{100000+i}" for i in range(N)],
        "email": [f"member{100000+i}@example.com" for i in range(N)],
        "phone": [f"555-01{rng.integers(0,100):02d}" for _ in range(N)],
        "membership_type": [rng.choice(tv[t]) for t in mtype],
        "join_date": join_date, "term_end_date": term_end, "guest_passes_used": guests,
        "status": [rng.choice(sv[c]) for c in churn_true]})
    pend = rng.random(N) < 0.015
    members.loc[pend, "status"] = rng.choice(["Pending", "Hold"], pend.sum())
    members.loc[rng.random(N) < 0.015, "membership_type"] = np.nan
    members.loc[rng.random(N) < 0.010, "term_end_date"] = pd.NaT
    members = pd.concat([members, members.sample(frac=0.03, random_state=SEED)], ignore_index=True)
    return members, visits

if CONFIG["USE_DEMO_DATA"]:
    members_raw, visits_raw = make_demo_data()
    members_raw.to_csv(OUT + "members_raw.csv", index=False); visits_raw.to_csv(OUT + "visits_raw.csv", index=False)
else:
    members_raw = pd.read_csv(CONFIG["MEMBERS_CSV"]); visits_raw = pd.read_csv(CONFIG["VISITS_CSV"])

# translate YOUR column names into the notebook's names
C = CONFIG["COLS"]
members_raw = members_raw.rename(columns={C["member_id"]: "member_id", C["plan"]: "plan", C["join_date"]: "join_date",
                                          C["term_end"]: "term_end", C["status"]: "status"})
vren = {C["member_id"]: "member_id", C["visit_date"]: "visit_date"}
if C.get("visit_id") and C["visit_id"] in visits_raw.columns: vren[C["visit_id"]] = "visit_id"
visits_raw = visits_raw.rename(columns=vren)
print("members_raw:", members_raw.shape, "| visits_raw:", visits_raw.shape)
members_raw.head()

members_raw: (4120, 8) | visits_raw: (339714, 3)


,member_id,email,phone,plan,join_date,term_end,guest_passes_used,status
0,M100000,member100000@example.com,555-0163,Monthly,2024-08-17,2025-07-29,1,Cancelled
1,M100001,member100001@example.com,555-0161,Month-to-Month,2024-03-22,2025-07-03,0,Active - Renewed
2,M100002,member100002@example.com,555-0166,Yearly,2024-05-05,2025-06-03,0,RENEWED
3,M100003,member100003@example.com,555-0152,Annual,2024-11-08,2025-12-25,1,RENEWED
4,M100004,member100004@example.com,555-0185,MONTHLY,2025-09-13,2026-01-30,1,cancelled


## 3. Data preparation, with a log of every change
| Step | Why |
|---|---|
| Drop columns that look like personal data (names, emails, phones, addresses) | Direct identifiers are never needed for modeling and must not be shared |
| Remove duplicate accounts and duplicate check-ins | Duplicates inflate counts and can leak across train/test. Real same-day visits are kept (matched by `visit_id`) |
| Tidy plan names and status text using your settings | The same plan or outcome is often typed several ways |
| Label missing plans `Unknown` | Keeps the member; missingness may itself matter |
| Drop members with no term-end date | The observation point cannot be defined |
| Set aside members whose status is unclear (Pending/Hold) | No honest label exists. They are **kept for scoring**, not training |
| Drop visits with no date | Cannot be placed on the timeline |
| **Keep only visits on or before the observation date** | Visits after the term ends are the future: using them is leakage |

In [4]:
members = members_raw.copy(); visits = visits_raw.copy()
LOG["members_raw_rows"] = len(members); LOG["visits_raw_rows"] = len(visits)

pii = [c for c in members.columns if c != "member_id" and any(p in c.lower() for p in CONFIG["PII_PATTERNS"])]
members = members.drop(columns=pii); LOG["personal_data_columns_dropped"] = pii

n = len(members); members = members.drop_duplicates(subset="member_id", keep="first")
LOG["duplicate_accounts_removed"] = n - len(members)
n = len(visits)
if "visit_id" in visits.columns:
    visits = visits.drop_duplicates(subset="visit_id")
else:
    LOG["warning"] = "No visit_id column: duplicate check-ins were NOT removed."
LOG["duplicate_visits_removed"] = n - len(visits)

def tidy_plan(x):
    if pd.isna(x): return "Unknown"
    t = str(x).strip().lower()
    for word, name in CONFIG["PLAN_WORDS"].items():
        if word in t: return name
    return "Other"
members["plan"] = members["plan"].map(tidy_plan)
LOG["missing_plan_filled_unknown"] = int((members["plan"] == "Unknown").sum())
LOG["unmatched_plan_set_other"] = int((members["plan"] == "Other").sum())

def label_status(x):
    t = str(x).strip().lower()
    if any(w in t for w in CONFIG["CHURNED_WORDS"]): return 1
    if any(w in t for w in CONFIG["RENEWED_WORDS"]): return 0
    return np.nan                                   # unclear -> not guessed
members["churned"] = members["status"].map(label_status)
for col in ("join_date", "term_end"): members[col] = pd.to_datetime(members[col], errors="coerce")

n = len(members); members = members.dropna(subset=["term_end", "join_date"]); LOG["dropped_missing_dates"] = n - len(members)
members["obs_date"] = members["term_end"] - pd.Timedelta(days=CONFIG["LEAD_DAYS"])
members_unres = members[members["churned"].isna()].copy()          # current/unclear members: scored later, not trained on
LOG["set_aside_unclear_status"] = len(members_unres)
members = members.dropna(subset=["churned"]); members["churned"] = members["churned"].astype(int)

visits["visit_date"] = pd.to_datetime(visits["visit_date"], errors="coerce")
n = len(visits); visits = visits.dropna(subset=["visit_date"]); LOG["visits_dropped_missing_date"] = n - len(visits)

chk = visits.merge(members[["member_id", "obs_date"]], on="member_id", how="inner")
LOG["visits_after_observation_excluded"] = int((chk["visit_date"] > chk["obs_date"]).sum())
LOG["members_clean_rows"] = len(members); LOG["churn_rate_clean"] = round(float(members["churned"].mean()), 4)
print(json.dumps(LOG, indent=2))

{
  "members_raw_rows": 4120,
  "visits_raw_rows": 339714,
  "personal_data_columns_dropped": [
    "email",
    "phone"
  ],
  "duplicate_accounts_removed": 120,
  "duplicate_visits_removed": 1690,
  "missing_plan_filled_unknown": 49,
  "unmatched_plan_set_other": 0,
  "dropped_missing_dates": 34,
  "set_aside_unclear_status": 46,
  "visits_dropped_missing_date": 1682,
  "visits_after_observation_excluded": 28625,
  "members_clean_rows": 3920,
  "churn_rate_clean": 0.2184
}


## 4. Features and data dictionary
All features use only information available on the observation date (term end minus `LEAD_DAYS`).

| Field | Definition |
|---|---|
| `days_since_last_visit` | Days from last check-in to the observation date (capped at 120; 120 also means no visits in the window) |
| `visits_per_month` | Check-ins in the 90 days before observation, divided by 3 |
| `attendance_trend` | Last-30-day visits ÷ average monthly visits in the prior 60 days (each +1 to avoid dividing by zero) |
| `tenure_months` | Months from join date to observation date |
| *optional numeric columns* | Any columns you list in `OPTIONAL_NUMERIC` |
| `plan` | Membership type (one-hot encoded) |
| `churned` | Target: 1 = did not renew, 0 = renewed |

In [5]:
OPT = [c for c in CONFIG["OPTIONAL_NUMERIC"] if c in members.columns]
FEATS = ["days_since_last_visit", "visits_per_month", "attendance_trend", "tenure_months"] + OPT

def build_features(m, v):
    vv = v.merge(m[["member_id", "obs_date"]], on="member_id", how="inner")
    vv = vv[vv["visit_date"] <= vv["obs_date"]]                    # never use the future
    vv = vv.assign(days_before=(vv["obs_date"] - vv["visit_date"]).dt.days)
    g = vv.groupby("member_id")["days_before"]
    f = pd.DataFrame({"days_since_last_visit": g.min().clip(upper=120),
                      "n90": g.apply(lambda s: (s < 90).sum()), "n30": g.apply(lambda s: (s < 30).sum()),
                      "n30_90": g.apply(lambda s: ((s >= 30) & (s < 90)).sum())})
    out = m.set_index("member_id").join(f)
    out["days_since_last_visit"] = out["days_since_last_visit"].fillna(120)
    for c_ in ["n90", "n30", "n30_90"]: out[c_] = out[c_].fillna(0)
    out["visits_per_month"] = out["n90"] / 3
    out["attendance_trend"] = ((out["n30"] + 1) / ((out["n30_90"] / 2) + 1)).clip(upper=5)
    out["tenure_months"] = (out["obs_date"] - out["join_date"]).dt.days / 30.4
    keep = FEATS + ["plan"] + (["churned"] if "churned" in out.columns else [])
    return out[keep]

df = build_features(members, visits)
df.to_csv(OUT + "members_clean.csv")
print(df.shape, "| churn rate:", round(df.churned.mean(), 3))
df.describe().round(2)

(3920, 7) | churn rate: 0.218


,days_since_last_visit,visits_per_month,attendance_trend,tenure_months,guest_passes_used,churned
count,3920.00,3920.00,3920.00,3920.00,3920.00,3920.00
mean,6.44,8.29,1.23,12.16,0.60,0.22
std,12.37,6.31,0.85,9.12,0.77,0.41
min,0.00,0.00,0.07,0.00,0.00,0.00
25%,1.00,3.67,0.67,5.33,0.00,0.00
50%,2.00,6.67,1.00,10.10,0.00,0.00
75%,7.00,11.33,1.50,16.78,1.00,0.00
max,120.00,45.00,5.00,58.22,5.00,1.00


## 5. Readiness report: read this before trusting anything below

In [6]:
n_rows, n_churn = len(df), int(df.churned.sum())
warnings = []
if n_rows < 500: warnings.append(f"Only {n_rows} members: results will be unstable. Aim for 500+.")
if n_churn < 100: warnings.append(f"Only {n_churn} churners: precision/recall estimates will be very noisy. Aim for 100+.")
rate = df.churned.mean()
if rate < 0.05 or rate > 0.60: warnings.append(f"Churn rate {rate:.0%} is extreme: check how you defined churn.")
if df["plan"].eq("Other").mean() > 0.10: warnings.append("Over 10% of plans are 'Other': extend PLAN_WORDS in the settings.")
if (df["days_since_last_visit"] == 120).mean() > 0.30: warnings.append("Over 30% of members show no visits in the window: check the visits export.")
READINESS = warnings if warnings else ["No warnings: data size and balance look workable."]
for w in READINESS: print("-", w)

- No warnings: data size and balance look workable.


## 6. Exploratory analysis

In [7]:
by_type = df.groupby("plan")["churned"].agg(churn_rate="mean", members="size").round(3)
bands = pd.cut(df["days_since_last_visit"], [-1, 7, 14, 30, 60, 121], labels=["0-7", "8-14", "15-30", "31-60", "61+"])
band_tbl = df.groupby(bands, observed=True)["churned"].agg(churn_rate="mean", members="size").round(3)
print(by_type, "\n"); print(band_tbl)

                churn_rate  members
plan                               
Annual               0.152     1154
Family               0.109      795
Monthly              0.317     1491
Student/Senior       0.258      431
Unknown              0.204       49 

                       churn_rate  members
days_since_last_visit                     
0-7                         0.188     3017
8-14                        0.277      458
15-30                       0.355      282
31-60                       0.377      122
61+                         0.415       41


## 7. Methods: model choice and justification
**Problem type.** Predict a yes/no outcome (renew or not) from member-level numbers, with a **minority positive class**
(often roughly one member in four or five churns). Staff need a ranked list, so the model must output a risk score, not only a label.

**Models and why.**
- **Majority-class baseline.** Predicts "renews" for everyone. It looks accurate (about 78%) while finding zero churners, which shows
  why accuracy is the wrong yardstick and gives the floor every real model must beat.
- **Logistic regression.** Interpretable and fast, with coefficients that can be turned into plain-language reasons for each flag.
- **Random forest.** Captures nonlinear thresholds and interactions with little tuning.
- **Gradient boosting (XGBoost).** Usually the strongest tabular learner; included to test whether extra complexity buys accuracy.

**Alternatives considered.** SMOTE oversampling was not used because class weights handled imbalance without inventing artificial members
(if used later it must be applied inside training folds only). A Cox survival model would estimate *time* to churn and is a planned extension.
Deep learning was rejected: small tables do not justify it and it is harder to explain to staff.

**Validation design.** 80/20 stratified split. On the 80% training portion, 5-fold stratified cross-validation scores each model and
chooses its decision threshold (best F1 on out-of-fold predictions). The test set is used once. Metrics: precision, recall, F1, PR-AUC
(preferred over ROC-AUC on imbalanced data), and a confusion matrix. The model is selected by **cross-validated PR-AUC**, never by test results.

In [8]:
X = pd.get_dummies(df.drop(columns="churned"), columns=["plan"]).astype(float)
y = df["churned"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
spw = (1 - y_tr.mean()) / y_tr.mean()

models = {
    "Majority baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)),
    "Random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced",
                                            random_state=SEED, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=250, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                             scale_pos_weight=spw, eval_metric="aucpr", random_state=SEED, n_jobs=2),
}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
results, fitted, test_scores = {}, {}, {}
for name, mdl in models.items():
    if name == "Majority baseline":
        thr, cvm, cvs = 0.5, float(y_tr.mean()), 0.0
    else:
        oof = cross_val_predict(mdl, X_tr, y_tr, cv=skf, method="predict_proba")[:, 1]
        p_, r_, t_ = precision_recall_curve(y_tr, oof)
        f1s = 2 * p_[:-1] * r_[:-1] / np.clip(p_[:-1] + r_[:-1], 1e-9, None)
        thr = float(t_[np.argmax(f1s)])
        sc = cross_validate(mdl, X_tr, y_tr, cv=skf, scoring="average_precision")["test_score"]
        cvm, cvs = float(sc.mean()), float(sc.std())
    mdl.fit(X_tr, y_tr)
    s = np.zeros(len(y_te)) if name == "Majority baseline" else mdl.predict_proba(X_te)[:, 1]
    pred = (s >= thr).astype(int) if name != "Majority baseline" else np.zeros(len(y_te), int)
    tn, fp, fn, tp = confusion_matrix(y_te, pred, labels=[0, 1]).ravel()
    results[name] = dict(cv_pr_auc=round(cvm, 3), cv_pr_auc_sd=round(cvs, 3), threshold=round(thr, 3),
                         test_pr_auc=round(float(average_precision_score(y_te, s)) if name != "Majority baseline" else float(y_te.mean()), 3),
                         precision=round(float(precision_score(y_te, pred, zero_division=0)), 3),
                         recall=round(float(recall_score(y_te, pred, zero_division=0)), 3),
                         f1=round(float(f1_score(y_te, pred, zero_division=0)), 3),
                         accuracy=round(float((pred == y_te).mean()), 3), tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))
    fitted[name], test_scores[name] = mdl, s

CHOSEN = max((k for k in results if k != "Majority baseline"), key=lambda k: results[k]["cv_pr_auc"])
print("Selected by cross-validated PR-AUC:", CHOSEN)
pd.DataFrame(results).T[["cv_pr_auc", "cv_pr_auc_sd", "test_pr_auc", "precision", "recall", "f1", "accuracy", "tp", "fp", "fn", "tn"]]

Selected by cross-validated PR-AUC: Logistic regression


,cv_pr_auc,cv_pr_auc_sd,test_pr_auc,precision,recall,f1,accuracy,tp,fp,fn,tn
Majority baseline,0.218,0.000,0.218,0.000,0.000,0.000,0.782,0.0,0.0,171.0,613.0
Logistic regression,0.404,0.016,0.329,0.305,0.696,0.424,0.588,119.0,271.0,52.0,342.0
Random forest,0.376,0.033,0.296,0.294,0.526,0.377,0.621,90.0,216.0,81.0,397.0
XGBoost,0.376,0.028,0.293,0.298,0.684,0.416,0.580,117.0,275.0,54.0,338.0


## 8. Fairness check and what drives the predictions
Look at the group table: if recall or precision differs a lot between groups, the list may serve some members better than others.
Decide with staff whether that is acceptable before using the list.

In [9]:
GROUP = CONFIG["FAIRNESS_GROUP"]
te = df.loc[X_te.index].copy()
te["score"] = test_scores[CHOSEN]; te["pred"] = (te["score"] >= results[CHOSEN]["threshold"]).astype(int)
type_err = te.groupby(GROUP).apply(
    lambda g: pd.Series({"test_members": len(g), "test_churners": int(g.churned.sum()),
                         "recall": recall_score(g.churned, g.pred, zero_division=0),
                         "precision": precision_score(g.churned, g.pred, zero_division=0)}), include_groups=False).round(3)
print(type_err)

pi = permutation_importance(fitted[CHOSEN], X_te, y_te, scoring="average_precision", n_repeats=10, random_state=SEED, n_jobs=1)
imp = pd.Series(pi.importances_mean, index=X.columns)
tcols = [c for c in X.columns if c.startswith("plan_")]
imp_g = imp.drop(tcols); imp_g["plan"] = imp[tcols].sum()
imp_g = imp_g.sort_values(ascending=False); print(imp_g.round(4))

                test_members  test_churners  recall  precision
plan                                                          
Annual                 207.0           39.0   0.333      0.325
Family                 171.0           19.0   0.316      0.316
Monthly                309.0           91.0   0.879      0.308
Student/Senior          86.0           20.0   0.900      0.286
Unknown                 11.0            2.0   1.000      0.250


visits_per_month         0.0377
plan                     0.0291
attendance_trend         0.0185
guest_passes_used        0.0096
days_since_last_visit    0.0051
tenure_months           -0.0036
dtype: float64


## 9. The at-risk list (the tool's main output)
Members whose outcome is not yet known (here: the Pending/Hold members set aside earlier; in real use, members approaching term end)
are scored with the selected model. Each flag comes with up to two plain-language reasons taken from the logistic regression.
**The list contains only the anonymous ID. Staff match IDs to people inside your own system. It is a prompt for a friendly check-in by a person.**

In [10]:
LABELS = {"days_since_last_visit": "Days since last visit", "visits_per_month": "Visit frequency",
          "attendance_trend": "Recent attendance trend", "tenure_months": "Time as a member"}
LABELS.update({c_: c_.replace("_", " ").capitalize() for c_ in OPT})
LABELS.update({c_: "Plan: " + c_[len("plan_"):] for c_ in X.columns if c_.startswith("plan_")})

to_score = build_features(members_unres, visits)
Xn = pd.get_dummies(to_score[FEATS + ["plan"]], columns=["plan"]).reindex(columns=X.columns, fill_value=0).astype(float)
risk = fitted[CHOSEN].predict_proba(Xn)[:, 1]

lr = fitted["Logistic regression"]
Z = lr.named_steps["standardscaler"].transform(Xn); contrib = Z * lr.named_steps["logisticregression"].coef_[0]
def reasons(i):
    order = np.argsort(-contrib[i])[:2]; out = []
    for j in order:
        if contrib[i, j] <= 0: continue
        name = X.columns[j]
        out.append(LABELS[name] if name.startswith("plan_") else f"{LABELS[name]} ({'above' if Z[i, j] > 0 else 'below'} typical)")
    return "; ".join(out)

at_risk = pd.DataFrame({"member_id": Xn.index, "plan": to_score["plan"].values, "risk_score": risk.round(2),
                        "flagged": (risk >= results[CHOSEN]["threshold"]),
                        "top_reasons": [reasons(i) for i in range(len(Xn))]}).sort_values("risk_score", ascending=False)
at_risk.to_csv(OUT + "at_risk_list.csv", index=False)
print(len(at_risk), "members scored;", int(at_risk.flagged.sum()), "flagged. Saved at_risk_list.csv")
at_risk.head(8)

46 members scored; 20 flagged. Saved at_risk_list.csv


,member_id,plan,risk_score,flagged,top_reasons
35,M102802,Monthly,0.82,True,Visit frequency (below typical); Plan: Monthly
4,M100368,Monthly,0.80,True,Visit frequency (below typical); Plan: Monthly
8,M100618,Monthly,0.78,True,Visit frequency (below typical); Plan: Monthly
11,M100880,Monthly,0.73,True,Visit frequency (below typical); Plan: Monthly
44,M103818,Monthly,0.73,True,Visit frequency (below typical); Plan: Monthly
21,M101915,Monthly,0.67,True,Plan: Monthly; Visit frequency (below typical)
38,M103241,Monthly,0.66,True,Visit frequency (below typical); Plan: Monthly
30,M102635,Monthly,0.66,True,Visit frequency (below typical); Plan: Monthly


## 10. Figures

In [11]:
BLUE, ORANGE, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "axes.edgecolor": GRID, "axes.labelcolor": MUTED,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True})
overall = df.churned.mean() * 100

order = by_type.drop(index="Unknown", errors="ignore").sort_values("churn_rate", ascending=False)
fig, ax = plt.subplots(figsize=(6.2, 3.1), dpi=200)
bars = ax.bar(order.index, order["churn_rate"] * 100, color=BLUE, width=0.55)
ax.axhline(overall, color=MUTED, lw=1, ls="--"); ax.text(len(order) - 0.55, overall + 0.8, f"All members {overall:.0f}%", ha="right", color=MUTED, fontsize=9)
for b, val in zip(bars, order["churn_rate"] * 100): ax.text(b.get_x() + b.get_width() / 2, val + 0.8, f"{val:.0f}%", ha="center", color=INK, fontsize=9)
ax.set_ylabel("Churn rate (%)"); ax.set_ylim(0, order["churn_rate"].max() * 100 + 8); ax.grid(axis="x", visible=False)
ax.set_title("Churn rate by plan (synthetic data)", loc="left", color=INK, fontsize=11)
fig.tight_layout(); fig.savefig(OUT + "fig1_churn_by_type.png"); plt.close(fig)

fig, ax = plt.subplots(figsize=(6.2, 3.1), dpi=200)
vals = band_tbl["churn_rate"].values * 100
bars = ax.bar(band_tbl.index.astype(str), vals, color=ORANGE, width=0.55)
for b, val in zip(bars, vals): ax.text(b.get_x() + b.get_width() / 2, val + 1, f"{val:.0f}%", ha="center", color=INK, fontsize=9)
ax.set_xlabel("Days since last visit (at observation date)"); ax.set_ylabel("Churn rate (%)"); ax.set_ylim(0, vals.max() + 12); ax.grid(axis="x", visible=False)
ax.set_title("Churn risk climbs the longer a member stays away", loc="left", color=INK, fontsize=11)
fig.tight_layout(); fig.savefig(OUT + "fig2_churn_by_recency.png"); plt.close(fig)

fig, ax = plt.subplots(figsize=(6.2, 3.4), dpi=200)
for name, c in {"Logistic regression": "#6250d6", "Random forest": "#1baf7a", "XGBoost": BLUE}.items():
    p_, r_, _ = precision_recall_curve(y_te, test_scores[name])
    ax.plot(r_, p_, color=c, lw=2, label=f"{name} (PR-AUC {results[name]['test_pr_auc']:.2f})")
ax.axhline(y_te.mean(), color=MUTED, lw=1, ls="--", label=f"Baseline = churn rate ({y_te.mean():.2f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_ylim(0, 1.02); ax.set_xlim(0, 1); ax.legend(frameon=False, fontsize=8.5, loc="upper right")
ax.set_title("All models beat the baseline, but only modestly (test set)", loc="left", color=INK, fontsize=11)
fig.tight_layout(); fig.savefig(OUT + "fig3_pr_curves.png"); plt.close(fig)

cm = np.array([[results[CHOSEN]["tn"], results[CHOSEN]["fp"]], [results[CHOSEN]["fn"], results[CHOSEN]["tp"]]])
fig, ax = plt.subplots(figsize=(4.6, 3.4), dpi=200); ax.grid(False)
ax.imshow(cm, cmap="Blues", vmin=0, vmax=cm.max() * 1.15)
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]}", ha="center", va="center", fontsize=14, color="white" if cm[i, j] > cm.max() * 0.55 else INK)
ax.set_xticks([0, 1], ["Predicted\nrenews", "Predicted\nchurns"]); ax.set_yticks([0, 1], ["Actually\nrenewed", "Actually\nchurned"])
for s_ in ax.spines.values(): s_.set_visible(False)
ax.set_title("Confusion matrix (test set)", loc="left", color=INK, fontsize=11)
fig.tight_layout(); fig.savefig(OUT + "fig4_confusion_matrix.png"); plt.close(fig)

s_ = imp_g.sort_values(); fig, ax = plt.subplots(figsize=(6.2, 3.1), dpi=200)
ax.barh([LABELS.get(i, i.replace("_", " ").capitalize()) if i != "plan" else "Membership plan" for i in s_.index], s_.values, color=BLUE, height=0.55)
ax.set_xlabel(f"Drop in PR-AUC when feature is shuffled ({CHOSEN}, test set)"); ax.grid(axis="y", visible=False)
ax.set_title("Which behaviors drive the predictions", loc="left", color=INK, fontsize=11)
fig.tight_layout(); fig.savefig(OUT + "fig5_importance.png"); plt.close(fig)
print("figures saved")

figures saved


In [12]:
summary = dict(cleaning_log=LOG, readiness=READINESS, n_members=int(len(df)), churn_rate=round(float(df.churned.mean()), 3),
               n_train=int(len(y_tr)), n_test=int(len(y_te)), test_churners=int(y_te.sum()), chosen=CHOSEN, results=results,
               n_scored=int(len(at_risk)), n_scored_flagged=int(at_risk.flagged.sum()),
               by_type=by_type.reset_index().rename(columns={"plan": "membership_type"}).to_dict("records"),
               type_error=type_err.reset_index().rename(columns={GROUP: "membership_type"}).to_dict("records"),
               importance={k: round(float(x), 4) for k, x in imp_g.items()},
               recency_bands={str(k): dict(churn_rate=float(r["churn_rate"]), n=int(r["members"])) for k, r in band_tbl.iterrows()})
json.dump(summary, open(OUT + "results.json", "w"), indent=2)
print("saved results.json")

saved results.json
